# SpikeData demo

Small walkthrough of the new `sim_data_analyzer.spike_data.SpikeData` API.

In [ ]:
import pickle
import sys
from pathlib import Path

import numpy as np

DIR_PACKAGE = Path.cwd().parent
DIR_REPO = DIR_PACKAGE.parent
if str(DIR_REPO) not in sys.path:
    sys.path.insert(0, str(DIR_REPO))

from sim_data_analyzer.spike_data import SpikeData


In [ ]:
# Parameters
FPATH_SIM_RESULT = (
    DIR_PACKAGE / 'dev_scratch' / 'data_src' / 'a1_lfp_15s' / 'data_00000_seed_1000.pkl'
)
DIRPATH_OUT = DIR_PACKAGE / 'dev_scratch' / 'data_proc' / 'a1_lfp_15s_0' / 'spike_data_demo'

T0 = 10.0
TMAX = 15.0
MS = True
NDIGITS = 3

FPATH_COMBINED = DIRPATH_OUT / 'spikes_combined_ms.npz'
FPATH_PER_CELL = DIRPATH_OUT / 'spikes_per_cell_ms.npz'

DIRPATH_OUT.mkdir(parents=True, exist_ok=True)
FPATH_SIM_RESULT


In [ ]:
# Load the NetPyNE simulation result
with FPATH_SIM_RESULT.open('rb') as fobj:
    sim_result = pickle.load(fobj)

sorted(sim_result['net']['pops'])[:10], len(sim_result['net']['pops'])


In [ ]:
# Extract combined spikes only
spikes_combined = SpikeData.from_sim_result(
    sim_result,
    combine=True,
    t0=T0,
    tmax=TMAX,
    subtract_t0=True,
    ms=MS,
    ndigits=NDIGITS,
)

print(spikes_combined.metadata)
print('First 8 populations:', spikes_combined.get_pop_names()[:8])
print('IT3 combined spikes:', spikes_combined.get_pop_spikes('IT3')[0][:20])


In [ ]:
# Save/load the combined representation
spikes_combined.save(FPATH_COMBINED)
spikes_combined_loaded = SpikeData.load(FPATH_COMBINED)

print(FPATH_COMBINED)
print(spikes_combined_loaded.metadata)
np.array_equal(
    spikes_combined.get_pop_spikes('IT3')[0],
    spikes_combined_loaded.get_pop_spikes('IT3')[0],
)


In [ ]:
# Extract per-cell spikes
spikes_per_cell = SpikeData.from_sim_result(
    sim_result,
    combine=False,
    t0=T0,
    tmax=TMAX,
    subtract_t0=True,
    ms=MS,
    ndigits=NDIGITS,
)

spikes_per_cell.save(FPATH_PER_CELL)
spikes_per_cell_loaded = SpikeData.load(FPATH_PER_CELL)

print(FPATH_PER_CELL)
print('IT3 cell gids:', spikes_per_cell_loaded.get_pop_cell_gids('IT3'))
print('IT3 number of cell trains:', len(spikes_per_cell_loaded.get_pop_spikes('IT3')))
print('IT3 first cell spikes:', spikes_per_cell_loaded.get_pop_spikes('IT3')[0][:20])


In [ ]:
# Combine per-cell spikes after loading
spikes_recombined = spikes_per_cell_loaded.combine()

same_it3 = np.array_equal(
    spikes_recombined.get_pop_spikes('IT3')[0],
    spikes_combined.get_pop_spikes('IT3')[0],
)

print('Recombined IT3 matches direct combined extraction:', same_it3)
print('Recombined IT3 spikes:', spikes_recombined.get_pop_spikes('IT3')[0][:20])
